# 10 · Backtracking

> Esplori un albero di scelte: **scegli**, **esplora** ricorsivamente, **annulla** la scelta. Serve quando devi elencare *tutte* le combinazioni, permutazioni, partizioni o configurazioni valide.
> Idee per i singoli problemi: [Backtracking e DP › Backtracking](../Fundamentals/Backtracking%20e%20DP.md#backtracking)

## Problemi → cosa ti serve

| Problema | Concetti | Sintassi |
|---|---|---|
| Subsets | per ogni elemento: prendo / non prendo | `path.append`, `path.pop`, `path[:]` |
| Combination Sum | riuso dello stesso elemento: ricorri con `i`, non `i + 1` | indice `start`, somma residua |
| Combination Sum II | ogni elemento una volta + input con duplicati | `candidates.sort()`, `if i > start and c[i] == c[i - 1]: continue` |
| Permutations | ogni posizione può prendere qualsiasi elemento non usato | `used` (set o lista di bool) |
| Subsets II | come Subsets, saltando i duplicati allo stesso livello | ordina + stesso `continue` di Combination Sum II |
| Word Search | backtracking su griglia | marcatura in-place, direzioni, bordi |
| Palindrome Partitioning | scegli dove finisce il prossimo pezzo | `s[start:end]`, `piece == piece[::-1]` |
| Letter Combinations of a Phone Number | un livello per cifra, un ramo per lettera | `dict` cifra → lettere, `"".join(path)` |
| N-Queens | una regina per riga, set di colonne e diagonali occupate | `r - c`, `r + c`, `"".join(row)` |

## Lo scheletro

```python
def backtrack(start, path):
    if <path è una soluzione completa>:
        result.append(path[:])   # COPIA
        return
    for i in range(start, len(items)):
        if <scelta non valida>: continue
        path.append(items[i])    # scegli
        backtrack(i + 1, path)   # esplora (i per riuso, i + 1 senza)
        path.pop()               # annulla
```

| Domanda | Cosa cambia |
|---|---|
| L'ordine conta? (permutazioni) | ciclo da `0` con `used`, invece che da `start` |
| Posso riusare un elemento? | ricorri con `i` invece di `i + 1` |
| L'input ha duplicati? | ordina e salta `items[i] == items[i - 1]` quando `i > start` |
| Tutti i nodi sono soluzioni? (subsets) | aggiungi `path[:]` a ogni chiamata, non solo alle foglie |

## Copiare il percorso

`path` è **una sola lista** modificata in tutta la ricorsione. Se aggiungi `path` al risultato, aggiungi un riferimento: alla fine vedrai solo liste vuote. Copia con `path[:]` o `list(path)`.

In [1]:
result_wrong, result_ok = [], []
path = []
for sku in ("MUG", "CAP"):
    path.append(sku)
    result_wrong.append(path)    # riferimento
    result_ok.append(path[:])    # copia
path.clear()
print(result_wrong, result_ok)  # -> [[], []] [['MUG'], ['MUG', 'CAP']]

[[], []] [['MUG'], ['MUG', 'CAP']]


## Combinazioni con indice di partenza

`start` impedisce di tornare indietro: ogni combinazione viene generata una sola volta, in ordine di indice. Qui: tutti i bundle di prodotti entro un budget (ogni prodotto al più una volta).

In [2]:
prices = [300, 450, 990]
budget = 800
bundles: list[list[int]] = []

def backtrack(start: int, path: list[int], remaining: int) -> None:
    bundles.append(path[:])  # ogni nodo è un bundle valido
    for i in range(start, len(prices)):
        if prices[i] > remaining:
            continue  # potatura: (con prezzi ordinati potresti usare break)
        path.append(prices[i])
        backtrack(i + 1, path, remaining - prices[i])
        path.pop()

backtrack(0, [], budget)
print(bundles)  # -> [[], [300], [300, 450], [450]]

[[], [300], [300, 450], [450]]


## Duplicati nell'input

Con `[1, 1, 2]` le scelte "primo 1" e "secondo 1" allo **stesso livello** producono gli stessi risultati. Ordinando, i duplicati sono adiacenti e li salti se `i > start` (il primo della serie a quel livello è ammesso).

In [3]:
items = sorted([2, 1, 1])
result = []

def backtrack(start: int, path: list[int]) -> None:
    result.append(path[:])
    for i in range(start, len(items)):
        if i > start and items[i] == items[i - 1]:
            continue
        path.append(items[i])
        backtrack(i + 1, path)
        path.pop()

backtrack(0, [])
print(result)  # -> [[], [1], [1, 1], [1, 1, 2], [1, 2], [2]]

[[], [1], [1, 1], [1, 1, 2], [1, 2], [2]]


## Permutazioni: `used`

Ogni posizione può prendere qualsiasi elemento non ancora usato, quindi il ciclo parte sempre da 0.

In [4]:
items = ["A", "B", "C"]
perms = []
used = [False] * len(items)

def backtrack(path: list[str]) -> None:
    if len(path) == len(items):
        perms.append("".join(path))
        return
    for i, item in enumerate(items):
        if used[i]:
            continue
        used[i] = True
        path.append(item)
        backtrack(path)
        path.pop()
        used[i] = False

backtrack([])
print(perms)  # -> ['ABC', 'ACB', 'BAC', 'BCA', 'CAB', 'CBA']

['ABC', 'ACB', 'BAC', 'BCA', 'CAB', 'CBA']


**Verifica rapida con `itertools`** (utile per testare la tua soluzione, non per consegnarla):

In [5]:
from itertools import combinations, permutations, product

print(list(combinations("ABC", 2)))                 # -> [('A', 'B'), ('A', 'C'), ('B', 'C')]
print(len(list(permutations("ABC"))))               # -> 6
print(["".join(p) for p in product("ab", "xy")])    # -> ['ax', 'ay', 'bx', 'by']

[('A', 'B'), ('A', 'C'), ('B', 'C')]
6
['ax', 'ay', 'bx', 'by']


## Un livello per posizione (stringhe)

Quando ogni posizione dell'output ha un insieme di scelte (lettere di una cifra, pezzi di una stringa), l'indice della ricorsione è la **posizione** nell'input.

In [6]:
letters = {"2": "abc", "3": "def"}
digits = "23"
out = []

def backtrack(i: int, path: list[str]) -> None:
    if i == len(digits):
        out.append("".join(path))
        return
    for ch in letters[digits[i]]:
        path.append(ch)
        backtrack(i + 1, path)
        path.pop()

backtrack(0, [])
print(out[:4], len(out))  # -> ['ad', 'ae', 'af', 'bd'] 9

['ad', 'ae', 'af', 'bd'] 9


**Partizioni:** dalla posizione `start`, prova ogni `end` e prendi il pezzo `s[start:end]`; ricorri da `end`. Controllo del palindromo: `piece == piece[::-1]`.

In [7]:
s = "aab"
start = 0
for end in range(start + 1, len(s) + 1):
    piece = s[start:end]
    print(piece, piece == piece[::-1])  # -> a True / aa True / aab False

a True
aa True
aab False


## Backtracking su griglia

Marca la cella come visitata modificandola (`"#"`), esplora i 4 vicini, poi ripristinala. Controlla i bordi **prima** di leggere la cella.

In [8]:
grid = [list("CAT"), list("XAR")]
rows, cols = len(grid), len(grid[0])

def spells(r: int, c: int, word: str, i: int = 0) -> bool:
    if i == len(word):
        return True
    if not (0 <= r < rows and 0 <= c < cols) or grid[r][c] != word[i]:
        return False
    saved, grid[r][c] = grid[r][c], "#"  # visitata
    found = any(spells(r + dr, c + dc, word, i + 1) for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)))
    grid[r][c] = saved                   # ripristino
    return found

print(spells(0, 0, "CAAR"), spells(0, 0, "CATA"), grid[0])  # -> True False ['C', 'A', 'T']

True False ['C', 'A', 'T']


## Vincoli con set: righe, colonne, diagonali

Su una scacchiera, le celle della stessa diagonale `↘` hanno lo stesso `r - c`; quelle dell'antidiagonale `↙` lo stesso `r + c`. Tre set ti dicono in `O(1)` se una cella è attaccata.

In [9]:
cols_used, diag, anti = set(), set(), set()
r, c = 1, 2
cols_used.add(c); diag.add(r - c); anti.add(r + c)

for cell in [(2, 3), (3, 0), (2, 0)]:
    rr, cc = cell
    attacked = cc in cols_used or (rr - cc) in diag or (rr + cc) in anti
    print(cell, attacked)  # -> (2, 3) True / (3, 0) True / (2, 0) False

board = [["."] * 4 for _ in range(4)]
board[1][2] = "Q"
print(["".join(row) for row in board])  # -> ['....', '..Q.', '....', '....']

(2, 3) True
(3, 0) True
(2, 0) False
['....', '..Q.', '....', '....']


## Errori comuni

- `result.append(path)` senza copia.
- Dimenticare di annullare la scelta (`pop`, `used[i] = False`, ripristino della cella, `remove` dai set).
- `if i > 0` invece di `if i > start` per i duplicati: perde combinazioni valide come `[1, 1]`.
- Complessità esponenziale: `O(2ⁿ)` sottoinsiemi, `O(n!)` permutazioni. È normale, i vincoli su `n` sono piccoli.